# PRICEX6
**Qwen catalog → SerpAPI Google Shopping → price parsing → cleaning → features → XGBoost → suggested range + minimum profitable price**

Uses the exact JSON structure returned by `generate_catalog()` in your uploaded `KARIGARKART_Sarvam_Qwen_Final(4).ipynb`. The two notebooks can run in separate kernels. No Sarvam key or Qwen model is copied into this notebook.

**Training:** fits XGBoost quantile regression to the current comparable listings and their advertised INR prices. It does not generate synthetic training prices or learn actual sale prices. The range estimates the 20th–80th conditional price quantiles, not guaranteed coverage or willingness to pay. A small per-product sample is a prototype; a larger, reviewed historical dataset is needed for validated pricing.

**Minimum profitable price:** calculated independently from your actual cost, desired **markup on cost** (default 18%) and percentage selling fee (default 0%). Include labour, packaging, shipping, fixed fees and applicable expenses in total cost. Qwen cannot know these costs. All costs and comparable prices must refer to the same sale unit/pack. If cost exceeds the estimated market range, show “No viable range” rather than inventing a profitable market price.

## Run in order
1. Run setup once; restart kernel if packages were installed.
2. Enter your SerpAPI key in the hidden prompt.
3. Provide the Qwen JSON and the actual total cost in **Cell 3**.
4. Run the remaining cells. The final cell prints only two lines.

Live searches may consume SerpAPI credits: at most three distinct queries per run. No automatic retry storm, no mock fallback. Debug tables and source records are available in the optional developer cell.

## 1. Setup — existing PyTorch/GPU packages are not needed

In [ ]:
import sys, subprocess, importlib.util
from importlib.metadata import version
from packaging.version import Version
requirements = {'numpy': 'numpy', 'pandas': 'pandas', 'requests': 'requests',
                'sklearn': 'scikit-learn', 'scipy': 'scipy', 'joblib': 'joblib'}
missing = [pkg for module,pkg in requirements.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec('xgboost') is None or Version(version('xgboost')) < Version('2.0'):
    missing.append('xgboost>=2.0,<4')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    print('Packages installed. Restart kernel before continuing.')
else:
    print('Setup complete.')

## 2. Imports and SerpAPI key

In [ ]:
import os, re, json, math, time, uuid
from datetime import datetime, timezone
from pathlib import Path
from getpass import getpass
from collections import Counter
import numpy as np
import pandas as pd
import requests
import joblib
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_pinball_loss
from xgboost import XGBRegressor

In [ ]:
SERPAPI_KEY = os.getenv('SERPAPI_KEY', '').strip()
if not SERPAPI_KEY:
    SERPAPI_KEY = getpass('Enter SerpAPI key (hidden): ').strip()
if not SERPAPI_KEY:
    raise ValueError('Key is empty. Run this cell again.')
print('SerpAPI key loaded.')

## 3. Qwen input and actual cost
In the Sarvam/Qwen dashboard, open **Structured result for backend → Catalog + supporting facts**. Save that JSON as `qwen_output.json` beside PRICEX6. Alternatively assign the full dictionary or a JSON string to `QWEN_OUTPUT` below. A plain English catalog dictionary is also accepted.

Expected full shape: `{"catalog": {"english": {"product_name": "...", "category": "...", "material": "...", "size": "...", "craft_technique": "...", "seo_description": "..."}, "hindi": {...}}, ...}`.

For automatic transfer on future generations, the **optional bridge at the end** can be pasted once into your existing Sarvam/Qwen notebook. No database is required.

Set `TOTAL_COST_INR` yourself, e.g. `700` only if ₹700 really is your total cost for this sale unit. `SALE_QUANTITY=1` means one item; set 2 for a pair, 6 for a six-piece pack, etc. Known mismatched bundles are excluded; listings that omit quantities still need review.

In [ ]:
# Paste Qwen's returned dict/JSON string here, OR leave None to read the JSON file.
QWEN_OUTPUT = None
QWEN_JSON_PATH = Path('qwen_output.json')
TOTAL_COST_INR = None       # Required: actual cost for the whole listing, e.g. 700
MARKUP_ON_COST = 0.18       # 18% markup, not 18% margin
SELLING_FEE_RATE = 0.0      # e.g. 0.10 if 10% of sale price goes to platform/payment fees
SALE_QUANTITY = 1
# Optional: a concise product type if Qwen's name is long/broad, e.g. 'jewellery box'.
PRODUCT_TYPE_OVERRIDE = ''
# Optional: primary material if Qwen lists many ingredients, e.g. 'lac'.
PRIMARY_MATERIAL_OVERRIDE = ''
MIN_COMPARABLES = 25        # Operational minimum, not a guarantee of accuracy
MIN_SELLERS = 3
MAX_SEARCHES = 3
FRESH_SEARCH = True
OUTPUT_DIR = Path('pricex6_runs')

## 4. Read Qwen output and build short search queries

In [ ]:
MISSING = {'', 'not specified', 'unknown', 'n/a', 'none', 'उल्लेख नहीं किया गया'}
ALIASES = {'wooden':'wood', 'jewellery':'jewelry', 'boxes':'box', 'bangles':'bangle',
           'earrings':'earring', 'necklaces':'necklace', 'vases':'vase', 'baskets':'basket',
           'sheesham':'wood', 'rosewood':'wood', 'terracotta':'clay', 'lacquer':'lac'}
STOP = set('a an the and of for with in by from to handmade handcrafted hand painted traditional premium beautiful unique artisan indian india natural decorative medium small large eco friendly'.split())
MATERIAL_WORDS = set('wood clay lac cotton silk wool jute bamboo brass copper silver gold iron steel leather glass stone bead beads mirror mirrors paper plastic resin ceramic'.split())

def tokens(text):
    return [ALIASES.get(w,w) for w in re.findall(r'[a-z]+',str(text).lower())]

def text_field(value):
    if not isinstance(value,str): return ''
    value=value.strip()
    return '' if value.lower() in MISSING else value

def normalize_qwen(output, product_type='', primary_material='', sale_quantity=1):
    if isinstance(output, str):
        try: output=json.loads(output)
        except ValueError: raise ValueError('Qwen input must be its structured JSON, not the displayed table.') from None
    if not isinstance(output,dict): raise ValueError('Provide Qwen output JSON/dictionary.')
    obj=output.get('catalog',output)
    if not isinstance(obj,dict): raise ValueError('catalog must be an object.')
    en=obj.get('english',obj)
    if not isinstance(en,dict): raise ValueError('catalog.english must be an object.')
    product={k:text_field(en.get(k,'')) for k in ('product_name','category','craft_technique','material','size','seo_description')}
    if not product['product_name']: raise ValueError('Qwen output has no English product_name.')
    if isinstance(sale_quantity,bool) or not isinstance(sale_quantity,int) or sale_quantity < 1:
        raise ValueError('SALE_QUANTITY must be a positive integer.')
    identity=list(dict.fromkeys(w for w in tokens(product_type or product['product_name']) if w not in STOP|MATERIAL_WORDS))
    if not identity: raise ValueError('Set PRODUCT_TYPE_OVERRIDE to a specific product type, e.g. jewellery box.')
    material=primary_material or re.split(r'[,;/]|\band\b',product['material'],maxsplit=1,flags=re.I)[0]
    material_tokens=list(dict.fromkeys(w for w in tokens(material) if w not in STOP))
    product.update(identity_tokens=identity, material_tokens=material_tokens, sale_quantity=sale_quantity)
    return product

def build_queries(product):
    # Never append the entire SEO paragraph. Remove repeats and progressively simplify.
    name=list(dict.fromkeys(tokens(product['product_name'])))
    identity=product['identity_tokens'];material=product['material_tokens']
    candidates=[' '.join(dict.fromkeys(name+material))[:160],
                ' '.join(dict.fromkeys(material+identity))[:160],
                ' '.join(identity)[:120]]
    return list(dict.fromkeys(q for q in candidates if q))[:MAX_SEARCHES]

def load_qwen_input():
    if QWEN_OUTPUT is not None: return QWEN_OUTPUT
    if not QWEN_JSON_PATH.is_file():
        raise ValueError('Save Structured result for backend as qwen_output.json here, or paste it into QWEN_OUTPUT in Cell 3.')
    return json.loads(QWEN_JSON_PATH.read_text(encoding='utf-8'))

## 5. SerpAPI — all supported shopping result blocks
Reads regular, inline and categorized results. No-results responses trigger the next shorter query. Key/quota errors stop immediately. Timeouts are reported without pretending they mean an invalid key. Provider errors are redacted before display; request URLs containing keys are never printed.

In [ ]:
def extract_rows(payload):
    rows=[]
    for field in ('shopping_results','inline_shopping_results'):
        items=payload.get(field) or []
        if isinstance(items,list): rows.extend(x for x in items if isinstance(x,dict))
    for group in payload.get('categorized_shopping_results') or []:
        if isinstance(group,dict):
            rows.extend(x for x in (group.get('shopping_results') or []) if isinstance(x,dict))
    return rows

def fetch_query(query, api_key):
    if not isinstance(api_key,str) or not api_key.strip(): raise ValueError('Enter your SerpAPI key first.')
    try:
        response=requests.get('https://serpapi.com/search.json',params={
            'engine':'google_shopping','q':query,'gl':'in','hl':'en',
            'api_key':api_key,'no_cache':'true' if FRESH_SEARCH else 'false'},timeout=(10,60))
    except requests.Timeout:
        raise RuntimeError('SerpAPI timed out while connecting/waiting. Try again later; no price was invented.') from None
    except requests.RequestException:
        raise RuntimeError('Could not connect to SerpAPI. Check connection and try again.') from None
    if response.status_code in (401,403): raise RuntimeError('SerpAPI authentication/access failed; check your key/account.')
    if response.status_code==429: raise RuntimeError('SerpAPI rate/quota limit; check your dashboard.')
    if not response.ok: raise RuntimeError(f'SerpAPI HTTP {response.status_code}; try later or check dashboard.')
    try: payload=response.json()
    except ValueError: raise RuntimeError('SerpAPI returned invalid JSON.') from None
    if not isinstance(payload,dict): raise RuntimeError('Unexpected SerpAPI response format.')
    error=str(payload.get('error') or '').replace(api_key,'[redacted]')
    no_results=bool(re.search(r"hasn't returned any results|no results|no shopping results",error,re.I))
    if error and not no_results: raise RuntimeError('SerpAPI: '+error[:350])
    meta=payload.get('search_metadata') or {}
    rows=extract_rows(payload)
    audit={'query':query,'received_utc':datetime.now(timezone.utc).isoformat(),
           'search_id':meta.get('id'),'provider_created_at':meta.get('created_at'),
           'provider_status':meta.get('status'),'raw_count':len(rows),
           'fresh_search_requested':FRESH_SEARCH,'no_results':no_results}
    # Keep only listing fields; no request URLs, raw headers or credentials in saved data.
    fields=('title','price','extracted_price','source','product_id','link','product_link','rating','reviews')
    return [{k:r.get(k) for k in fields} for r in rows],audit

## 6. Price parsing and market cleaning
Only explicit INR prices, positive finite values, matching product-type keywords and primary material pass. Duplicate offers/titles are removed. Known pack mismatches, used listings, installment prices and ambiguous price ranges are rejected. Wide statistical outliers are removed using log-price IQR when enough rows exist.

Search titles do not reliably contain dimensions or craft technique: missing size/pack information is recorded for review, not guessed. “Wood” does not distinguish sheesham from other wood; check comparable titles in the optional debug cell. These filters reduce noise but cannot establish visual equivalence or authenticity.

In [ ]:
def parse_price(row):
    raw=str(row.get('price') or '').strip()
    if not re.search(r'₹|\bINR\b|\bRs\.?',raw,re.I): return None
    if re.search(r'[$€£]|\bUSD\b|\bEUR\b|\bGBP\b|/month|\bEMI\b|\bfrom\b|\bstarting\b|[-–—]',raw,re.I): return None
    nums=re.findall(r'\d[\d,]*(?:\.\d+)?',raw)
    if len(nums)!=1: return None
    parsed=float(nums[0].replace(',',''))
    extracted=row.get('extracted_price')
    if extracted is not None:
        try: numeric=float(extracted)
        except (TypeError,ValueError): return None
        if not np.isfinite(numeric) or abs(numeric-parsed)>0.05: return None
    return parsed if np.isfinite(parsed) and parsed>0 else None

def listing_quantity(title):
    for pattern in (r'(?:pack|set)\s+of\s+(\d+)',r'(\d+)\s*(?:pcs|pieces|piece|pc)\b'):
        match=re.search(pattern,title,re.I)
        if match:return int(match.group(1))
    if re.search(r'\bpair\b',title,re.I):return 2
    return None

def clean_market_data(rows,product):
    kept=[]; rejected=Counter()
    for row in rows:
        title=str(row.get('title') or '').strip()
        price=parse_price(row)
        if not title or price is None: rejected['invalid/non-INR/ambiguous price']+=1;continue
        words=set(tokens(title))
        if not set(product['identity_tokens']).issubset(words): rejected['product type mismatch']+=1;continue
        if product['material_tokens'] and not set(product['material_tokens']).issubset(words):
            rejected['material mismatch or not stated']+=1;continue
        if re.search(r'\bused\b|\bpre.?owned\b|\brefurbished\b',title,re.I):rejected['used product']+=1;continue
        qty=listing_quantity(title)
        if qty is not None and qty!=product['sale_quantity']:rejected['pack mismatch']+=1;continue
        # Reject unknown bundle counts; an unspecified standalone title is an assumed single item.
        if qty is None and (product['sale_quantity']!=1 or re.search(r'\bset\b|\bpack\b|\bcombo\b',title,re.I)):
            rejected['unknown pack quantity']+=1;continue
        kept.append({'title':title,'price':price,'seller':str(row.get('source') or 'Unknown'),
                     'product_id':str(row.get('product_id') or ''),
                     'link':row.get('product_link') or row.get('link'),
                     'quantity_explicit':qty is not None,'normalized_title':' '.join(tokens(title))})
    cols=['title','price','seller','product_id','link','quantity_explicit','normalized_title']
    frame=pd.DataFrame(kept,columns=cols)
    before=len(frame)
    frame=frame.drop_duplicates('normalized_title')
    # A product ID may appear with different titles across result blocks/queries.
    mask=frame.product_id.ne('') & frame.product_id.duplicated()
    frame=frame[~mask].copy();rejected['duplicates']=before-len(frame)
    if len(frame)>=10:
        logs=np.log(frame.price.to_numpy(dtype=float));q1,q3=np.quantile(logs,[.25,.75]);iqr=q3-q1
        if iqr>0:
            keep=(logs>=q1-1.5*iqr)&(logs<=q3+1.5*iqr)
            rejected['log-price outlier']=int((~keep).sum());frame=frame.loc[keep]
    return frame.reset_index(drop=True),dict(rejected)

def seller_count(frame):
    return frame.loc[frame.seller.str.lower().ne('unknown'),'seller'].str.lower().nunique()

def fetch_market(product,api_key):
    raw=[];searches=[]
    for query in build_queries(product):
        rows,audit=fetch_query(query,api_key)
        raw.extend(rows);searches.append(audit)
        clean,rejected=clean_market_data(raw,product)
        if len(clean)>=MIN_COMPARABLES and seller_count(clean)>=MIN_SELLERS:break
    return raw,clean,rejected,searches

## 7. Feature engineering and XGBoost training
**X:** listing-title TF-IDF plus product/material keyword coverage and title length. **y:** log of the actual fetched listing price. No competitor production costs are invented. Target-product text uses Qwen product name, material and craft technique; SEO descriptions are not used as price labels.

One XGBoost model predicts two quantiles (20%, 80%). A fixed holdout checks listing-price quantile loss and coverage for developer inspection only; it is too small and biased to represent real sales accuracy. The final model is fitted to all reviewed automatic comparables. Market quantiles bound extrapolation; crossing predictions are sorted. Models use CPU, not your laptop GPU.

In [ ]:
def feature_text(text):
    return ' '.join(tokens(text))

def engineered_features(texts,product,vectorizer,fit=False):
    clean=[feature_text(t) for t in texts]
    text_x=vectorizer.fit_transform(clean) if fit else vectorizer.transform(clean)
    extra=[]
    identity=set(product['identity_tokens']); material=set(product['material_tokens'])
    for t in texts:
        w=set(tokens(t))
        extra.append([len(tokens(t))/30, len(w&identity)/max(1,len(identity)),
                      len(w&material)/max(1,len(material))])
    return hstack([text_x,csr_matrix(np.asarray(extra,dtype=np.float32))],format='csr')

def make_model():
    return XGBRegressor(objective='reg:quantileerror',quantile_alpha=np.array([0.2,0.8]),
        n_estimators=180,max_depth=3,learning_rate=0.04,tree_method='hist',
        min_child_weight=3,subsample=0.9,colsample_bytree=0.9,
        random_state=42,n_jobs=2)

def train_xgboost(frame,product):
    if len(frame)<MIN_COMPARABLES or seller_count(frame)<MIN_SELLERS:
        raise ValueError('Not enough comparable listings to train XGBoost.')
    texts=frame.title.tolist(); y=np.log(frame.price.to_numpy(dtype=float))
    train,test=train_test_split(np.arange(len(frame)),test_size=.2,random_state=42)
    vectorizer=TfidfVectorizer(ngram_range=(1,2),max_features=128,sublinear_tf=True)
    X_train=engineered_features([texts[i] for i in train],product,vectorizer,True)
    X_test=engineered_features([texts[i] for i in test],product,vectorizer)
    validation_model=make_model();validation_model.fit(X_train,y[train])
    predicted=np.sort(np.exp(validation_model.predict(X_test)),axis=1)
    actual=np.exp(y[test])
    diagnostics={'target':'advertised INR listing price, NOT actual sale price',
        'train_rows':len(train),'holdout_rows':len(test),
        'pinball_q20_INR':float(mean_pinball_loss(actual,predicted[:,0],alpha=.2)),
        'pinball_q80_INR':float(mean_pinball_loss(actual,predicted[:,1],alpha=.8)),
        'holdout_interval_coverage':float(np.mean((actual>=predicted[:,0])&(actual<=predicted[:,1]))),
        'caution':'Small search-selected sample; not production accuracy or calibrated coverage.'}
    final_vec=TfidfVectorizer(ngram_range=(1,2),max_features=128,sublinear_tf=True)
    X_all=engineered_features(texts,product,final_vec,True)
    model=make_model();model.fit(X_all,y)
    target=' '.join(product[k] for k in ('product_name','material','craft_technique') if product[k])
    X_target=engineered_features([target],product,final_vec)
    limits=np.sort(np.exp(model.predict(X_target)[0]))
    if not np.isfinite(limits).all(): raise RuntimeError('Non-finite XGBoost output.')
    # Restrict extrapolation beyond the observed central market distribution.
    observed=np.quantile(frame.price.to_numpy(dtype=float),[.1,.9])
    limits=np.clip(limits,*observed)
    return limits,model,final_vec,diagnostics

## 8. Minimum profitable price and final range — two outputs only

In [ ]:
def minimum_price(total_cost,markup=0.18,fee_rate=0.0):
    values=[total_cost,markup,fee_rate]
    if any(isinstance(v,bool) or not isinstance(v,(int,float,np.number)) or not np.isfinite(v) for v in values):
        raise ValueError('Enter your actual TOTAL_COST_INR as a number in Cell 3.')
    if total_cost<=0 or markup<0 or not 0<=fee_rate<1:
        raise ValueError('Cost must be positive, markup nonnegative, fee rate between 0 and 1.')
    return math.ceil(total_cost*(1+markup)/(1-fee_rate))

def predict_price(qwen_result,total_cost,api_key,markup=0.18,fee_rate=0.0,
                  sale_quantity=1,product_type='',primary_material='',save=True):
    floor=minimum_price(total_cost,markup,fee_rate)
    product=normalize_qwen(qwen_result,product_type,primary_material,sale_quantity)
    raw,frame,rejected,searches=fetch_market(product,api_key)
    audit={'product':product,'source':'SerpAPI Google Shopping listings',
           'searches':searches,'raw_count':len(raw),'cleaned_count':len(frame),
           'rejections':rejected,'total_cost_INR':float(total_cost),
           'markup_on_cost':markup,'selling_fee_rate':fee_rate,
           'quantity_warning':'Unstated single-item quantities are assumed; review dimensions and pack units.'}
    model=vectorizer=None
    public={'suggested_price_range':None,'minimum_profitable_price':floor}
    if len(frame)<MIN_COMPARABLES or seller_count(frame)<MIN_SELLERS:
        audit['status']='insufficient_comparables'
    else:
        limits,model,vectorizer,diagnostics=train_xgboost(frame,product)
        low=math.ceil(max(float(limits[0]),floor));high=math.floor(float(limits[1]))
        audit.update(model_quantiles_INR=[float(x) for x in limits],diagnostics=diagnostics)
        if low>high:
            audit['status']='cost_above_range' if floor>float(limits[1]) else 'range_too_narrow'
        else:
            public['suggested_price_range']=[low,high];audit['status']='ok'
    if save:
        folder=OUTPUT_DIR/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
        folder.mkdir(parents=True,exist_ok=False)
        (folder/'result.json').write_text(json.dumps(public,ensure_ascii=False,indent=2),encoding='utf-8')
        (folder/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
        (folder/'listings.json').write_text(json.dumps(raw,ensure_ascii=False,indent=2),encoding='utf-8')
        (folder/'cleaned_listings.json').write_text(frame.to_json(orient='records',force_ascii=False,indent=2),encoding='utf-8')
        if model is not None:
            model.save_model(folder/'xgboost.ubj')
            joblib.dump(vectorizer,folder/'title_vectorizer.joblib')
        audit['saved_folder']=str(folder.resolve())
    return public,audit,frame

def show_result(result,audit):
    interval=result['suggested_price_range']
    if interval is not None:message=f'₹{interval[0]:,} – ₹{interval[1]:,}'
    else:message={'insufficient_comparables':'Unavailable — too few comparable products found.',
                  'cost_above_range':'No viable range — costs exceed estimated market prices.',
                  'range_too_narrow':'Unavailable — estimated range too narrow.'}.get(audit['status'],'Unavailable')
    print('Suggested price range     : '+message)
    print(f"Minimum profitable price  : ₹{result['minimum_profitable_price']:,}")

## 9. Run PRICEX6

In [ ]:
# Re-run this cell after changing Qwen input or costs; it may make up to 3 searches.
PRICEX6_RESULT = PRICEX6_AUDIT = PRICEX6_MARKET = None
try:
    PRICEX6_RESULT, PRICEX6_AUDIT, PRICEX6_MARKET = predict_price(
        load_qwen_input(),TOTAL_COST_INR,SERPAPI_KEY,
        markup=MARKUP_ON_COST,fee_rate=SELLING_FEE_RATE,sale_quantity=SALE_QUANTITY,
        product_type=PRODUCT_TYPE_OVERRIDE,primary_material=PRIMARY_MATERIAL_OVERRIDE)
    show_result(PRICEX6_RESULT,PRICEX6_AUDIT)
except (ValueError,RuntimeError) as error:
    print(str(error))

## 10. Optional developer inspection — off by default

In [ ]:
SHOW_DEBUG = False
if SHOW_DEBUG and PRICEX6_AUDIT is not None:
    from IPython.display import display
    display(PRICEX6_MARKET)
    display(PRICEX6_AUDIT)

## Optional automatic Qwen → PRICEX6 handoff
Paste this small cell **into your existing Sarvam/Qwen notebook after the Qwen functions cell**, then generate a new catalog using its existing dashboard. It wraps the current `generate_catalog` function, so the existing dashboard calls save the full supported result automatically. If you re-run the original Qwen functions cell, re-run this bridge afterward. Keep both notebooks in the same folder, or set the same absolute JSON path in both.

```python
from pathlib import Path
import json, threading, uuid

if not getattr(generate_catalog, '_pricex6_bridge', False):
    _generate_for_pricex6 = generate_catalog
    _pricex6_export_lock = threading.Lock()

    def generate_catalog(text):
        result = _generate_for_pricex6(text)
        with _pricex6_export_lock:
            destination = Path('qwen_output.json').resolve()
            temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.tmp')
            try:
                temporary.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
                temporary.replace(destination)
            finally:
                temporary.unlink(missing_ok=True)
        return result

    generate_catalog._pricex6_bridge = True
```

This is a latest-result handoff for a single user; re-run PRICEX6's final cell after each new catalog. For a future combined backend, pass the returned dictionary directly to `predict_price(qwen_result, total_cost, serpapi_key)` instead of using a shared file. Avoid importing/running the entire audio notebook as a module because it starts the microphone/dashboard and credentials flow.

## Validation and limits
Code and the full processing path were tested locally with **explicit test fixtures and actual XGBoost training**, including Qwen schema parsing, all three result blocks, INR parsing, missing material, bundles, duplicates, insufficient data, API errors, timeouts and cost-floor conflicts. Test fixtures are not used in this delivered notebook. Live SerpAPI fetching was not executed here; enter your own key to verify live results on your machine. No live price output or real-world accuracy is claimed.

The automatic filter may reject legitimate sparse listings or admit wrong sizes/finishes; manually review matches before production. A default 18% markup is an editable business assumption, not evidence of demand or guaranteed profit. Model/vectorizer artifacts saved by this notebook are specific to each product run; only load your own trusted joblib files.

Sources: [SerpAPI Shopping response format](https://serpapi.com/google-shopping-api), [XGBoost parameters: quantile objective](https://xgboost.readthedocs.io/en/stable/parameter.html).